In [ ]:
from pathlib import Path
from PIL import Image


for category in ["bedroom", "church"]:
    INPUT_DIR = Path(f"lsun/data/{category}")
    OUTPUT_DIR = Path(f"data/{category}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    images = list(INPUT_DIR.rglob("*.webp"))

    for i, image_path in enumerate(images):
        with Image.open(image_path) as img:
            img.convert("RGB").save(OUTPUT_DIR / f"{i:05d}.png")

    print(f"Converted {len(images)} images.")
    print(f"Saved to: {OUTPUT_DIR}")

Converted 300 images.
Saved to: data/bedroom
Converted 300 images.
Saved to: data/church


In [ ]:
from pathlib import Path
from PIL import Image
import random

random.seed(7)

INPUT_DIRS = [
    Path("data/bedroom"),
    Path("data/church")
    Path("data/celeba/")
]

OUTPUT_ROOT = Path("../data/")
NUM_SAMPLES = 100
CROP_SIZE = 256

for input_dir in INPUT_DIRS:
    images = sorted(input_dir.rglob("*.jpg"))

    if len(images) < NUM_SAMPLES:
        raise ValueError(f"Not enough images in {input_dir}")

    sampled_images = random.sample(images, NUM_SAMPLES)

    output_dir = OUTPUT_ROOT / "celeba" #input_dir.name
    output_dir.mkdir(parents=True, exist_ok=True)

    for i, image_path in enumerate(sampled_images):
        with Image.open(image_path) as img:
            img = img.convert("RGB")

            width, height = img.size

            if width < CROP_SIZE or height < CROP_SIZE:
                scale = CROP_SIZE / min(width, height)

                new_width = round(width * scale)
                new_height = round(height * scale)
                img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

                width, height = img.size

            # Center crop only — no resizing or interpolation
            left = (width - CROP_SIZE) // 2
            top = (height - CROP_SIZE) // 2
            right = left + CROP_SIZE
            bottom = top + CROP_SIZE

            img = img.crop((left, top, right, bottom))
            img.save(output_dir / f"{i:05d}.png")

    print(f"{input_dir.name}: saved {NUM_SAMPLES} center-cropped images")

print("Done!")

img_align_celeba: saved 100 center-cropped images
Done!


In [1]:
from datasets import load_dataset

ds = load_dataset("blanchon/UC_Merced")

/home/issa/anaconda3/envs/weak_prior/lib/python3.12/importlib/__init__.py:90: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.4)
  return _bootstrap._gcd_import(name[level:], package, level)


In [6]:
from datasets import load_dataset
import os

UC_Merced = load_dataset("blanchon/UC_Merced")

output_dir = "data/UC_Merced"

for split in UC_Merced:
    for idx, sample in enumerate(UC_Merced[split]):
        label = sample["label"]
        image = sample["image"]

        class_name = UC_Merced[split].features["label"].names[label]

        class_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        image.save(os.path.join(class_dir, f"{split}_{idx:04d}.jpg"))

print("Done!")

Done!


In [2]:
from torchvision.datasets import CelebA

dataset = CelebA(
    root="./data",
    split="test",
    download=True
)

In [ ]:

from pathlib import Path

from datasets import load_dataset
from tqdm import tqdm


# Configuration
DATASET_NAME = "timm/resisc45"
SPLIT = "validation"
OUTPUT_DIR = Path("../data/resisc45/")



OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading {DATASET_NAME} ({SPLIT})...")

# Hugging Face downloads the requested split
dataset = load_dataset(
    DATASET_NAME,
    split=SPLIT,
)

print(f"Found {len(dataset)} images")
print(f"Features: {dataset.features}")

# Extract class names from the dataset's ClassLabel feature
label_feature = dataset.features["label"]

for sample in tqdm(dataset, desc="Saving images"):
    image = sample["image"]
    label = label_feature.int2str(sample["label"])
    image_id = sample["image_id"]

    # Create class directory
    class_dir = OUTPUT_DIR / label
    class_dir.mkdir(parents=True, exist_ok=True)

    # Save image
    output_path = class_dir / f"{image_id}.png"

    if output_path.exists():
        continue

    image.convert("RGB").save(
        output_path,
        format="png",
        quality=95,
    )

print("\nDownload complete!")
print(f"Images saved to: {OUTPUT_DIR}")

# Count downloaded images
total_images = sum(
    1 for path in OUTPUT_DIR.rglob("*.PNG")
)

print(f"Total images: {total_images}")
print(f"Total classes: {len(list(OUTPUT_DIR.iterdir()))}")

Loading timm/resisc45 (validation)...


/home/issa/anaconda3/envs/weak_prior/lib/python3.12/importlib/__init__.py:90: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.4)
  return _bootstrap._gcd_import(name[level:], package, level)


Found 6300 images
Features: {'image': Image(mode=None, decode=True, id=None), 'label': ClassLabel(names=['airplane', 'airport', 'baseball_diamond', 'basketball_court', 'beach', 'bridge', 'chaparral', 'church', 'circular_farmland', 'cloud', 'commercial_area', 'dense_residential', 'desert', 'forest', 'freeway', 'golf_course', 'ground_track_field', 'harbor', 'industrial_area', 'intersection', 'island', 'lake', 'meadow', 'medium_residential', 'mobile_home_park', 'mountain', 'overpass', 'palace', 'parking_lot', 'railway', 'railway_station', 'rectangular_farmland', 'river', 'roundabout', 'runway', 'sea_ice', 'ship', 'snowberg', 'sparse_residential', 'stadium', 'storage_tank', 'tennis_court', 'terrace', 'thermal_power_station', 'wetland'], id=None), 'image_id': Value(dtype='string', id=None)}


Saving images:  35%|███▌      | 2221/6300 [00:26<00:48, 83.87it/s]


KeyboardInterrupt: 